# Lab 1: Music Genre Classification
Este notebook contiene la solución completa para las Tareas 1 y 2 del laboratorio de clasificación de géneros musicales.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

def evaluate_models(X, y):
    """
    Entrena y evalúa múltiples modelos usando 10-fold CV y ajuste de hiperparámetros. Después busca la mejor configuración de cada algoritmo.
    """
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    
    models = {
        'Logistic Regression': (LogisticRegression(max_iter=1000), {'C': [0.1, 1, 10]}),
        'k-NN': (KNeighborsClassifier(), {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}),
        'Decision Tree': (DecisionTreeClassifier(random_state=42), {'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}),
        'Random Forest': (RandomForestClassifier(random_state=42), {'n_estimators': [50, 100], 'max_depth': [None, 10]}),
        'ANN': (MLPClassifier(max_iter=500, random_state=42), {'hidden_layer_sizes': [(50,), (100,)], 'alpha': [0.0001, 0.001]})
    }
    
    results = {}
    
    for name, (model, params) in models.items():
        print(f"Entrenando {name}...")
        grid = GridSearchCV(model, params, cv=cv, scoring='accuracy', n_jobs=-1)
        grid.fit(X, y)
        results[name] = {
            'Mejor Accuracy': grid.best_score_,
            'Mejores Parametros': grid.best_params_
        }
        print(f"Accuracy: {grid.best_score_:.4f} con {grid.best_params_}\n")
        
    return pd.DataFrame(results).T

## Task 1: Dataset GenreAll

In [4]:
# Cargar y procesar GenreAll.csv
df_genre_all = pd.read_csv('GenreAll.csv', skipinitialspace = True)

X_all = df_genre_all.drop(columns=['genre'])
y_all = df_genre_all['genre']

X_all = pd.get_dummies(X_all, drop_first=True)
le = LabelEncoder()
y_all_encoded = le.fit_transform(y_all)

scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X_all)

selector = SelectKBest(score_func=f_classif, k=50)
X_all_selected = selector.fit_transform(X_all_scaled, y_all_encoded)

print("--- Resultados para GenreAll.csv ---")
results_task1 = evaluate_models(X_all_selected, y_all_encoded)
display(results_task1)

--- Resultados para GenreAll.csv ---
Entrenando Logistic Regression...
Accuracy: 0.7673 con {'C': 1}

Entrenando k-NN...
Accuracy: 0.7410 con {'n_neighbors': 5, 'weights': 'distance'}

Entrenando Decision Tree...
Accuracy: 0.5658 con {'max_depth': 10, 'min_samples_split': 2}

Entrenando Random Forest...
Accuracy: 0.7350 con {'max_depth': None, 'n_estimators': 100}

Entrenando ANN...
Accuracy: 0.7955 con {'alpha': 0.001, 'hidden_layer_sizes': (50,)}



,Mejor Accuracy,Mejores Parametros
Logistic Regression,0.767313,{'C': 1}
k-NN,0.74103,"{'n_neighbors': 5, 'weights': 'distance'}"
Decision Tree,0.565838,"{'max_depth': 10, 'min_samples_split': 2}"
Random Forest,0.735,"{'max_depth': None, 'n_estimators': 100}"
ANN,0.795535,"{'alpha': 0.001, 'hidden_layer_sizes': (50,)}"


## Task 2: Dataset features_30_sec

In [5]:
# Cargar y procesar features_30_sec.csv
df_30sec = pd.read_csv('features_30_sec.csv')

if 'filename' in df_30sec.columns:
    df_30sec = df_30sec.drop(columns=['filename'])

X_30 = df_30sec.drop(columns=['label'])
y_30 = df_30sec['label']

y_30_encoded = le.fit_transform(y_30)
X_30_scaled = scaler.fit_transform(X_30)

print("--- Resultados para features_30_sec.csv ---")
results_task2_30sec = evaluate_models(X_30_scaled, y_30_encoded)
display(results_task2_30sec)

--- Resultados para features_30_sec.csv ---
Entrenando Logistic Regression...
Accuracy: 0.7370 con {'C': 1}

Entrenando k-NN...
Accuracy: 0.7180 con {'n_neighbors': 7, 'weights': 'distance'}

Entrenando Decision Tree...
Accuracy: 0.6060 con {'max_depth': 10, 'min_samples_split': 5}

Entrenando Random Forest...
Accuracy: 0.7940 con {'max_depth': 10, 'n_estimators': 100}

Entrenando ANN...
Accuracy: 0.7520 con {'alpha': 0.0001, 'hidden_layer_sizes': (100,)}



,Mejor Accuracy,Mejores Parametros
Logistic Regression,0.737,{'C': 1}
k-NN,0.718,"{'n_neighbors': 7, 'weights': 'distance'}"
Decision Tree,0.606,"{'max_depth': 10, 'min_samples_split': 5}"
Random Forest,0.794,"{'max_depth': 10, 'n_estimators': 100}"
ANN,0.752,"{'alpha': 0.0001, 'hidden_layer_sizes': (100,)}"


In [ ]:
# Unir los resultados de ambos datasets
tabla_resumen = results_task1.join(
    results_task2_30sec, 
    lsuffix=' (GenreAll)', 
    rsuffix=' (30_sec)'
)

# Mostrar la tabla resumen
display(tabla_resumen)

,Mejor Accuracy (GenreAll),Mejores Parametros (GenreAll),Mejor Accuracy (30_sec),Mejores Parametros (30_sec)
Logistic Regression,0.767313,{'C': 1},0.737,{'C': 1}
k-NN,0.74103,"{'n_neighbors': 5, 'weights': 'distance'}",0.718,"{'n_neighbors': 7, 'weights': 'distance'}"
Decision Tree,0.565838,"{'max_depth': 10, 'min_samples_split': 2}",0.606,"{'max_depth': 10, 'min_samples_split': 5}"
Random Forest,0.735,"{'max_depth': None, 'n_estimators': 100}",0.794,"{'max_depth': 10, 'n_estimators': 100}"
ANN,0.795535,"{'alpha': 0.001, 'hidden_layer_sizes': (50,)}",0.752,"{'alpha': 0.0001, 'hidden_layer_sizes': (100,)}"


5. ¿Es real o está inflada la precisión de 3 segundos?
Está inflada por la fuga de datos (data leakage). Al dividir una pista en fragmentos, la validación cruzada estándar coloca partes de la misma canción tanto en entrenamiento como en prueba. El modelo memoriza las canciones en lugar de aprender el género.

1. ¿Son 3 segundos suficientes para identificar un género?
Sí, en la mayoría de los casos. Las "texturas" como el timbre y la instrumentación se capturan casi de inmediato. Sin embargo, fallan en géneros con dinámicas variables o evolución lenta (como clásica o jazz), donde 3 segundos pueden capturar, por ejemplo, un silencio.

11. ¿Qué familias de características importan más?
Las espectrales (MFCC, centroide, rolloff) son las más determinantes porque definen la "huella dactilar" de los instrumentos y el timbre. Las rítmicas (BPM, tasa de onsets) también son importantes para aislar géneros dependientes de la percusión (hip-hop, electrónica).

15. ¿Está bien planteada la clasificación con una sola etiqueta (single-label)?
No. La música moderna fusiona múltiples estilos (ej. pop-rock), haciendo que forzar una única etiqueta sea subjetivo y confunda al algoritmo ante pistas acústicamente idénticas.